In [1]:
import sys
import os
from pathlib import Path

In [3]:
import gc
gc.collect()

0

In [20]:
from pathlib import Path
from langchain_docling import DoclingLoader
from langchain_core.documents import Document
from tqdm import tqdm
import re 

# Cleaning OF 3GPPG docs
def remove_preface(text:str)->str:
    """
    Remove the content before:
    'Definitions, Symbols and Abbreviations'
    """
    pattern = r"(?:3\s+Definitions,\s*symbols\s*and\s*abbreviations|Definitions,\s*symbols\s*and\s*abbreviations)"
    
    matches = list(re.finditer(pattern, text, flags=re.IGNORECASE))
    
    if len(matches) >= 2:
        return text[matches[1].start():]
    return text

# Normalisation on all_docs
def normalise_text(text:str)->str:
    # replace tabs with spaces
    text = text.replace('\t', ' ')
    # replace multiple spaces with a single space
    text = re.sub(r' +', ' ', text)
    # remove excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)
    # strip leading and trailing whitespace
    text = "\n".join(line.strip() for line in text.splitlines())
    return text.strip()

# chunk the docs by sections 
def chunk_by_sections(text:str)->list:
    '''
    Chunk text based on section headers like '4.1', '4.1.1', '4.2', etc.
    Returns list of chunks with their section numbers.
    '''
    section_pattern = r'^(\d+(?:\.\d+)*)\s+([A-Z][A-Za-z\s\-]+)$'
    chunks = []
    lines = text.split('\n')
    current_section = 'Preamble'  # Default section for content before the first header
    current_chunk = []
    
    for line in lines:
        match = re.match(section_pattern, line.strip())
        if match:
            # If we have a current chunk, save it before starting a new one
            if current_chunk:
                chunk_text = '\n'.join(current_chunk).strip()
                if chunk_text:  # Only add non-empty chunks
                    chunks.append(
                                  {
                            'section': current_section,
                            'content': chunk_text
                        }
                    )
                current_chunk = []
            current_section = match.group(1)  # e.g., '4.1', '4.1.1'
            current_chunk.append(line)  # Include the section header in the chunk
        else:
            if current_section:  # Only add lines if we are within a section
                current_chunk.append(line)
    if current_chunk:  # Add the last chunk if it exists
        chunk_text = '\n'.join(current_chunk).strip()
        if chunk_text:  # Only add non-empty chunks 
            chunks.append(
                {
                    'section': current_section,
                    'content': '\n'.join(current_chunk).strip()
                }
            )   
    print(f"Chunked into {len(chunks)} sections.")
            
    return chunks   

def chunk_with_hierarchy(text: str, min_chunk_size: int = 250) -> list:
    """
    Advanced chunking that maintains hierarchy:
    - If a section is too large, try to split into subsections
    - If still too large, split by paragraphs within the subsection
    """
    sections = chunk_by_sections(text)
    final_chunks = []
    
    for section in sections:
        content = section['content']
        section_num = section['section']
        
        word_count = len(content.split())
        if word_count <= min_chunk_size:
            final_chunks.append(Document(page_content=content, metadata={'section': section_num}))
        else:
            # Try to split into subsections
            # Check if there are subsections (next level of numbering)
            sub_pattern = re.compile(rf"^{re.escape(section_num)}\.\d+\s+", re.MULTILINE)
            sub_matches = list(sub_pattern.finditer(content))
            
            if sub_matches:
                # Split by subsections
                splits = sub_pattern.split(content)
                # The first element is text before first subsection (should be section header)
                if splits[0].strip():
                    final_chunks.append(Document(
                        page_content=splits[0].strip(),
                        metadata={
                            'section': section_num,
                            'subsection': 'intro',
                            'word_count': len(splits[0].split())
                        }
                    ))
                
                # Process each subsection
                for i, sub_content in enumerate(splits[1:], 1):
                    if sub_content.strip():
                        final_chunks.append(Document(
                            page_content=sub_content.strip(),
                            metadata={
                                'section': section_num,
                                'subsection': f"{section_num}.{i}",
                                'word_count': len(sub_content.split())
                            }
                        ))
            else:
                # No subsections found, split by paragraphs
                paragraphs = content.split('\n\n')
                current_paragraphs = []
                current_size = 0
                
                for para in paragraphs:
                    para_size = len(para.split())
                    
                    if current_size + para_size > min_chunk_size and current_paragraphs:
                        chunk_text = '\n\n'.join(current_paragraphs)
                        final_chunks.append(Document(
                            page_content=chunk_text,
                            metadata={
                                'section': section_num,
                                'word_count': len(chunk_text.split())
                            }
                        ))
                        current_paragraphs = [para]
                        current_size = para_size
                    else:
                        current_paragraphs.append(para)
                        current_size += para_size
                
                # Add remaining paragraphs
                if current_paragraphs:
                    chunk_text = '\n\n'.join(current_paragraphs)
                    final_chunks.append(Document(
                        page_content=chunk_text,
                        metadata={
                            'section': section_num,
                            'word_count': len(chunk_text.split())
                        }
                    ))
    
    return final_chunks
    

# 3GPP docs

# Find project root dynamically
PROJECT_ROOT = Path.cwd().parent
# Build paths safely
three_gpp_dir = PROJECT_ROOT / "data" / "raw" / "3gpp_docs/"

all_docs = []

for doc_file in tqdm(three_gpp_dir.glob("37340-h30.docx"), desc="Processing 3GPP documents"):
    try : 
        loader = DoclingLoader(str(doc_file))
        for doc in loader.lazy_load():
            doc.page_content = remove_preface(doc.page_content)
            doc.page_content = normalise_text(doc.page_content)
            
            print(f"Processed document: {doc_file}, content length: {len(doc.page_content)}")
            # Apply section-based chunking
            chunks = chunk_with_hierarchy(doc.page_content, min_chunk_size=200)
            # Add document-level metadata to each chunk
            for chunk in chunks:
                chunk.metadata.update({
                    'source': str(doc_file),
                    'doc_name': doc_file.name
                })
            
            all_docs.extend(chunks)
            print(f"Created {len(chunks)} chunks from document")
            
    except Exception as e:
        print(f"Error processing {doc_file}: {e}")
        
print(f"Total documents loaded: {len(all_docs)}")





Processing 3GPP documents: 0it [00:00, ?it/s][transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (868 > 512). Running this sequence through the model will result in indexing errors
Processing 3GPP documents: 1it [00:16, 16.45s/it]

Processed document: c:\Users\ankit\Desktop\Telecom_Rag\data\raw\3gpp_docs\37340-h30.docx, content length: 1056
Chunked into 2 sections.
Created 2 chunks from document
Processed document: c:\Users\ankit\Desktop\Telecom_Rag\data\raw\3gpp_docs\37340-h30.docx, content length: 616
Chunked into 1 sections.
Created 1 chunks from document
Processed document: c:\Users\ankit\Desktop\Telecom_Rag\data\raw\3gpp_docs\37340-h30.docx, content length: 727
Chunked into 1 sections.
Created 1 chunks from document
Processed document: c:\Users\ankit\Desktop\Telecom_Rag\data\raw\3gpp_docs\37340-h30.docx, content length: 856
Chunked into 1 sections.
Created 1 chunks from document
Processed document: c:\Users\ankit\Desktop\Telecom_Rag\data\raw\3gpp_docs\37340-h30.docx, content length: 683
Chunked into 1 sections.
Created 1 chunks from document
Processed document: c:\Users\ankit\Desktop\Telecom_Rag\data\raw\3gpp_docs\37340-h30.docx, content length: 802
Chunked into 1 sections.
Created 1 chunks from document
Pro

In [ ]:
# Embedding Process 
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS


if len(all_docs) > 0:
    print('Embedding Start')
    
    model_name = 'BAAI/bge-small-en'
    model_kwargs = {
        'device': 'cpu'
    }
    encode_kwargs = {
        'normalize_embeddings': True
    }
    
    embeddings = HuggingFaceEmbeddings(model_name=model_name, model_kwargs=model_kwargs, encode_kwargs=encode_kwargs)
    
    # create faiss vector store
    vectorstore = FAISS.from_documents(
        documents=all_docs,
        embedding=embeddings
    )
    
    # Save the vector store to disk
    save_path = PROJECT_ROOT/'data'/'vectorstore'/'faiss_index'
    save_path.parent.mkdir(parents=True, exist_ok=True)  # Ensure the directory exists
    vectorstore.save_local(str(save_path))
    print(f"Vector store saved to {save_path}")
    

    
    


Embedding Start


c:\Users\ankit\Desktop\Telecom_Rag\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ankit\.cache\huggingface\hub\models--BAAI--bge-small-en. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3232.01it/s]


Vector store saved to c:\Users\ankit\Desktop\Telecom_Rag\data\vectorstore\faiss_index


In [ ]:
all_docs 

In [45]:
# Importing
from  pathlib import Path
from langchain_docling import DoclingLoader
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS 
import re
from tqdm import tqdm 

# ─────────────────────────────────────────────────────────────────────────────
#  CONSTANTS
# ─────────────────────────────────────────────────────────────────────────────

MIN_CHUNK_WORDS      = 50    # chunks below this are discarded as noise
DEFAULT_CHUNK_SIZE   = 200   # target min words per chunk
PARENT_CONTEXT_LINES = 3     # how many lines of parent section to prepend to child chunks


# ============================================================================
# TEXT CLEANING FUNCTIONS
# ============================================================================



# ─────────────────────────────────────────────────────────────────────────────
#  STEP 1 — CLEANING
# ─────────────────────────────────────────────────────────────────────────────

def remove_preface(text: str) -> str:
    """
    Remove boilerplate before 'Definitions, Symbols and Abbreviations'.
    Looks for the SECOND occurrence because the first is in the Table of Contents.
    """
    pattern = (
        r"(?:3\s+Definitions,\s*symbols\s*and\s*abbreviations"
        r"|Definitions,\s*symbols\s*and\s*abbreviations)"
    )
    matches = list(re.finditer(pattern, text, flags=re.IGNORECASE))
    if len(matches) >= 2:
        return text[matches[1].start():]
    elif len(matches) == 1:
        # Only one match found — still better to start from there than keep boilerplate
        return text[matches[0].start():]
    return text


def remove_void_sections(text: str) -> str:
    """
    Remove 'Void' placeholder sections that 3GPP uses when a clause was deleted.
    These add noise to the vector store.
    Example:  '5.3.2  Void'
    """
    return re.sub(
        r'^\d+(?:\.\d+)*\s+Void\s*$',
        '',
        text,
        flags=re.IGNORECASE | re.MULTILINE
    )


def normalise_text(text: str) -> str:
    """
    Standard whitespace normalisation.
    """
    text = text.replace('\t', ' ')
    text = re.sub(r' +', ' ', text)                                # collapse spaces
    text = re.sub(r"\n{3,}", "\n\n", text)                         # max 2 blank lines
    text = "\n".join(line.strip() for line in text.splitlines())   # strip each line
    return text.strip()


def clean_document(text: str) -> str:
    """
    Master cleaning function — run all cleaning steps in order.
    """
    text = remove_preface(text)
    text = remove_void_sections(text)
    text = normalise_text(text)
    return text


# ─────────────────────────────────────────────────────────────────────────────
#  STEP 2 — SECTION DETECTION
# ─────────────────────────────────────────────────────────────────────────────

# FIX: Loosened regex compared to original.
# Original required title to start with capital and contain only letters/spaces/hyphens.
# This missed headers like:
#   "5.3.4 UE behaviour in RRC_IDLE"  (underscore)
#   "6.1.2 NR and E-UTRA"             (hyphen mid-word)
#   "7.2 TS 38.331 reference"         (spec number in title)
#
# New pattern:
#   - Section number: one or more digit groups separated by dots  e.g. 5, 5.3, 5.3.4
#   - At least one space
#   - Title: anything that starts with a letter (upper or lower), min 3 chars
#   - Anchored to start of line (MULTILINE)

SECTION_PATTERN = re.compile(
    r'^(\d+(?:\.\d+)*)\s{1,4}([A-Za-z].{2,}?)$',
    re.MULTILINE
)


def is_valid_section_title(title: str) -> bool:
    """
    Extra guard to reject false-positive section matches.
    Filters out things like pure number lines, very short noise lines,
    and common false positives from table formatting.
    """
    title = title.strip()
    if len(title) < 3:
        return False
    if title.lower() in ('void', 'n/a', 'reserved', 'tbd', 'ffs'):
        return False
    # Reject lines that are mostly numbers/symbols (likely table rows)
    alpha_ratio = sum(c.isalpha() for c in title) / max(len(title), 1)
    if alpha_ratio < 0.4:
        return False
    return True


def find_sections(text: str) -> list[dict]:
    """
    Find all section headers in the text.
    Returns list of dicts:
        { 'num': '5.3.1', 'title': 'UE behaviour', 'start': <char_pos> }
    """
    sections = []
    for match in SECTION_PATTERN.finditer(text):
        num   = match.group(1).strip()
        title = match.group(2).strip()
        if is_valid_section_title(title):
            sections.append({
                'num':   num,
                'title': title,
                'start': match.start()
            })
    return sections


# ─────────────────────────────────────────────────────────────────────────────
#  STEP 3 — CHUNKING
# ─────────────────────────────────────────────────────────────────────────────

def get_parent_context(section_num: str, all_sections: list[dict], full_text: str) -> str:
    """
    FIX: Original code had no parent context injection.
    
    For a child section like '5.3.2', find its parent '5.3' and extract
    the first PARENT_CONTEXT_LINES lines as a context header to prepend.
    
    This makes every chunk self-contained — the LLM knows what the parent
    clause is about even when only seeing the child chunk.
    """
    parts = section_num.split('.')
    if len(parts) <= 1:
        return ''  # top-level section has no parent

    parent_num = '.'.join(parts[:-1])

    # Find parent section in the list
    parent = next((s for s in all_sections if s['num'] == parent_num), None)
    if not parent:
        return ''

    # Find where the parent section's content starts and ends
    parent_idx = all_sections.index(parent)
    parent_start = parent['start']
    # Parent ends where the next section starts (or end of text)
    if parent_idx + 1 < len(all_sections):
        parent_end = all_sections[parent_idx + 1]['start']
    else:
        parent_end = len(full_text)

    parent_content = full_text[parent_start:parent_end].strip()
    parent_lines   = [l for l in parent_content.splitlines() if l.strip()]

    if not parent_lines:
        return ''

    # Take heading + first few content lines as context
    context_lines = parent_lines[:PARENT_CONTEXT_LINES]
    return f"[Parent: §{parent_num} — {parent['title']}]\n" + "\n".join(context_lines)


def split_into_section_chunks(full_text: str) -> list[dict]:
    """
    Split the full document text into per-section raw chunks.
    Each chunk contains the complete text of one section.
    
    Returns list of:
        { 'num': '5.3', 'title': 'Header text', 'content': '...' }
    """
    sections = find_sections(full_text)
    if not sections:
        # Fallback: treat entire document as one chunk
        return [{'num': '0', 'title': 'Full Document', 'content': full_text}]

    raw_chunks = []
    for i, section in enumerate(sections):
        start = section['start']
        end   = sections[i + 1]['start'] if i + 1 < len(sections) else len(full_text)

        content = full_text[start:end].strip()
        if content:
            raw_chunks.append({
                'num':     section['num'],
                'title':   section['title'],
                'content': content
            })

    return raw_chunks


def get_section_depth(section_num: str) -> int:
    """Return depth of section: '5' → 1, '5.3' → 2, '5.3.1' → 3"""
    return len(section_num.split('.'))


def chunk_large_section_by_paragraphs(
    content: str,
    section_num: str,
    section_title: str,
    parent_context: str,
    min_chunk_size: int
) -> list[Document]:
    """
    Last-resort splitter: when a section has no subsections and is too large,
    split by paragraphs and group until we reach min_chunk_size.
    """
    paragraphs    = [p.strip() for p in content.split('\n\n') if p.strip()]
    chunks        = []
    current_paras = []
    current_words = 0
    part_num      = 1

    for para in paragraphs:
        para_words = len(para.split())
        if current_words + para_words > min_chunk_size and current_paras:
            chunk_text = assemble_chunk(
                parent_context,
                '\n\n'.join(current_paras)
            )
            chunks.append(Document(
                page_content=chunk_text,
                metadata={
                    'section':       section_num,
                    'section_title': section_title,
                    'part':          part_num,
                    'word_count':    len(chunk_text.split()),
                    'chunk_type':    'paragraph_split'
                }
            ))
            current_paras = [para]
            current_words = para_words
            part_num     += 1
        else:
            current_paras.append(para)
            current_words += para_words

    # Remaining paragraphs
    if current_paras:
        chunk_text = assemble_chunk(parent_context, '\n\n'.join(current_paras))
        chunks.append(Document(
            page_content=chunk_text,
            metadata={
                'section':       section_num,
                'section_title': section_title,
                'part':          part_num,
                'word_count':    len(chunk_text.split()),
                'chunk_type':    'paragraph_split'
            }
        ))

    return chunks


def assemble_chunk(parent_context: str, body: str) -> str:
    """
    Combine optional parent context header with the chunk body.
    """
    if parent_context:
        return f"{parent_context}\n\n{body}"
    return body


def chunk_with_hierarchy(
    full_text: str,
    min_chunk_size: int = DEFAULT_CHUNK_SIZE
) -> list[Document]:
    """
    Main chunking function.
    
    Strategy:
    1. Split text into per-section raw blocks using section headers.
    2. For each block:
       a. If small enough → emit as one chunk with parent context prepended.
       b. If large and has subsections → split at subsections (preserving their headers).
       c. If large and no subsections → split by paragraphs.
    3. Filter out any chunk below MIN_CHUNK_WORDS.
    
    FIX vs original:
    - Section headers are NO LONGER lost when splitting (was a bug in original).
    - Subsection metadata uses REAL section numbers, not enumerate index.
    - Parent context is injected into every child chunk.
    - Minimum word count filter removes noise chunks.
    """
    all_sections = find_sections(full_text)
    raw_chunks   = split_into_section_chunks(full_text)
    final_docs   = []

    for raw in raw_chunks:
        section_num   = raw['num']
        section_title = raw['title']
        content       = raw['content']
        word_count    = len(content.split())

        parent_context = get_parent_context(section_num, all_sections, full_text)

        # ── Case A: Small enough — emit as single chunk ──────────────────
        if word_count <= min_chunk_size:
            chunk_text = assemble_chunk(parent_context, content)
            if len(chunk_text.split()) >= MIN_CHUNK_WORDS:
                final_docs.append(Document(
                    page_content=chunk_text,
                    metadata={
                        'section':       section_num,
                        'section_title': section_title,
                        'word_count':    len(chunk_text.split()),
                        'chunk_type':    'full_section'
                    }
                ))
            continue

        # ── Case B: Large — try splitting by direct subsections ──────────
        # We look for subsections that are exactly ONE level deeper.
        # e.g., for section '5.3', we look for '5.3.X' but NOT '5.3.X.Y'
        current_depth    = get_section_depth(section_num)
        subsec_pattern   = re.compile(
            rf'^({re.escape(section_num)}\.\d+)\s{{1,4}}([A-Za-z].{{2,}}?)$',
            re.MULTILINE
        )
        subsec_matches   = list(subsec_pattern.finditer(content))

        if subsec_matches:
            # FIX: Use finditer + manual slicing to PRESERVE subsection headers.
            # Original used re.split() which consumed (discarded) the matched header text.
            split_positions = [m.start() for m in subsec_matches] + [len(content)]

            # Text before the first subsection (intro paragraph of parent)
            intro = content[:split_positions[0]].strip()
            if intro and len(intro.split()) >= MIN_CHUNK_WORDS:
                chunk_text = assemble_chunk(parent_context, intro)
                final_docs.append(Document(
                    page_content=chunk_text,
                    metadata={
                        'section':       section_num,
                        'section_title': section_title,
                        'subsection':    'intro',
                        'word_count':    len(chunk_text.split()),
                        'chunk_type':    'subsection_intro'
                    }
                ))

            # Each subsection — slice from its start to next subsection start
            for i, match in enumerate(subsec_matches):
                sub_start = split_positions[i]
                sub_end   = split_positions[i + 1]
                sub_text  = content[sub_start:sub_end].strip()

                # FIX: Use the REAL section number from the regex match,
                # not an enumerate index like the original did.
                real_sub_num   = match.group(1).strip()   # e.g. '5.3.2'
                real_sub_title = match.group(2).strip()   # e.g. 'UE behaviour'

                if not sub_text or len(sub_text.split()) < MIN_CHUNK_WORDS:
                    continue

                sub_parent_ctx = get_parent_context(real_sub_num, all_sections, full_text)
                sub_word_count = len(sub_text.split())

                if sub_word_count <= min_chunk_size:
                    chunk_text = assemble_chunk(sub_parent_ctx, sub_text)
                    final_docs.append(Document(
                        page_content=chunk_text,
                        metadata={
                            'section':       real_sub_num,
                            'section_title': real_sub_title,
                            'word_count':    len(chunk_text.split()),
                            'chunk_type':    'subsection'
                        }
                    ))
                else:
                    # Subsection itself is large → paragraph-split it
                    para_chunks = chunk_large_section_by_paragraphs(
                        sub_text, real_sub_num, real_sub_title,
                        sub_parent_ctx, min_chunk_size
                    )
                    final_docs.extend(para_chunks)

        else:
            # ── Case C: No subsections — split by paragraphs ─────────────
            para_chunks = chunk_large_section_by_paragraphs(
                content, section_num, section_title,
                parent_context, min_chunk_size
            )
            final_docs.extend(para_chunks)

    print(f"  → {len(final_docs)} chunks after filtering (min {MIN_CHUNK_WORDS} words)")
    return final_docs


# ─────────────────────────────────────────────────────────────────────────────
#  STEP 4 — DOCUMENT LOADING & ORCHESTRATION
# ─────────────────────────────────────────────────────────────────────────────

def load_and_chunk_3gpp_docs(
    three_gpp_dir: Path,
    glob_pattern: str = "*.docx",       # FIX: was hardcoded to one file
    min_chunk_size: int = DEFAULT_CHUNK_SIZE
) -> list[Document]:
    """
    Load all 3GPP documents from a directory, clean, chunk, and return
    a flat list of LangChain Documents with full metadata.

    Args:
        three_gpp_dir:  Path to folder containing 3GPP .docx files.
        glob_pattern:   File pattern to match. Default '*.docx' processes all.
                        Use e.g. '37340-h30.docx' for a single file during testing.
        min_chunk_size: Target minimum words per chunk.
    """
    doc_files = list(three_gpp_dir.glob(glob_pattern))
    if not doc_files:
        print(f"[WARNING] No files found in {three_gpp_dir} matching '{glob_pattern}'")
        return []

    all_docs = []

    for doc_file in tqdm(doc_files, desc="Processing 3GPP documents"):
        try:
            loader = DoclingLoader(str(doc_file))
            raw_pages = list(loader.lazy_load())

            if not raw_pages:
                print(f"  [SKIP] {doc_file.name} — DoclingLoader returned no content")
                continue

            # Merge all pages into one text block
            # (DoclingLoader may split a single docx across multiple Document objects)
            full_text = "\n\n".join(p.page_content for p in raw_pages if p.page_content)

            print(f"\n[INFO] {doc_file.name} — raw length: {len(full_text):,} chars")

            # Clean
            full_text = clean_document(full_text)
            print(f"After cleaning: {len(full_text):,} chars")

            # Chunk
            chunks = chunk_with_hierarchy(full_text, min_chunk_size=min_chunk_size)

            # Attach document-level metadata to every chunk
            for chunk in chunks:
                chunk.metadata.update({
                    'source':   str(doc_file),
                    'doc_name': doc_file.name,
                    'doc_type': '3gpp_spec'
                })

            all_docs.extend(chunks)
            print(f"Created {len(chunks)} chunks")

        except Exception as e:
            print(f"  [ERROR] Failed to process {doc_file}: {e}")
            import traceback
            traceback.print_exc()

    print(f"\n{'─'*60}")
    print(f"Total 3GPP chunks ready for embedding: {len(all_docs)}")
    print(f"{'─'*60}")
    return all_docs


# ─────────────────────────────────────────────────────────────────────────────
#  STEP 5 — QUICK VALIDATION
# ─────────────────────────────────────────────────────────────────────────────

def validate_chunks(docs: list[Document], sample_size: int = 10) -> None:
    """
    Spot-check the output chunks.
    Prints statistics and a sample of chunks to review manually.
    Run this after loading to catch any remaining issues before embedding.
    """
    if not docs:
        print("[VALIDATE] No documents to validate.")
        return

    word_counts = [d.metadata.get('word_count', len(d.page_content.split())) for d in docs]
    chunk_types = {}
    for d in docs:
        ct = d.metadata.get('chunk_type', 'unknown')
        chunk_types[ct] = chunk_types.get(ct, 0) + 1

    print("\n── Chunk Validation Report ──────────────────────────────────")
    print(f"Total chunks      : {len(docs)}")
    print(f"Min words/chunk   : {min(word_counts)}")
    print(f"Max words/chunk   : {max(word_counts)}")
    print(f"Avg words/chunk   : {sum(word_counts) / len(word_counts):.0f}")
    print(f"Chunk type breakdown:")
    for ct, count in sorted(chunk_types.items(), key=lambda x: -x[1]):
        print(f"  {ct:<25} {count}")

    # Check for chunks that slipped through with suspiciously low word count
    tiny = [d for d in docs if len(d.page_content.split()) < MIN_CHUNK_WORDS]
    if tiny:
        print(f"\n[WARNING] {len(tiny)} chunks below {MIN_CHUNK_WORDS} words — review these:")
        for t in tiny[:5]:
            print(f"  §{t.metadata.get('section')} — {len(t.page_content.split())} words")

    # Check metadata completeness
    missing_section = [d for d in docs if 'section' not in d.metadata]
    if missing_section:
        print(f"\n[WARNING] {len(missing_section)} chunks missing 'section' metadata")

    # Sample output
    import random
    print(f"\n── Sample Chunks (random {sample_size}) ──────────────────────────")
    for d in random.sample(docs, min(sample_size, len(docs))):
        print(f"\n  §{d.metadata.get('section', 'N/A')} | {d.metadata.get('section_title', '')} "
              f"| {d.metadata.get('word_count', '?')} words | {d.metadata.get('chunk_type', '?')}")
        preview = d.page_content[:200].replace('\n', ' ')
        print(f"  Preview: {preview}...")
    print("─────────────────────────────────────────────────────────────\n")


# ─────────────────────────────────────────────────────────────────────────────
#  ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":

    PROJECT_ROOT  = Path.cwd().parent
    three_gpp_dir = PROJECT_ROOT / "data" / "raw" / "3gpp_docs"

    # ── For testing: process only one file ──
    # all_docs = load_and_chunk_3gpp_docs(three_gpp_dir, glob_pattern="37340-h30.docx")

    
    all_docs = load_and_chunk_3gpp_docs(
        three_gpp_dir,
        glob_pattern="*.docx",
        min_chunk_size=DEFAULT_CHUNK_SIZE
    )

    # Validate output before passing to embedding
    validate_chunks(all_docs, sample_size=10)

    # ── Next step: pass all_docs to your embedding + FAISS pipeline ──
    # from embedding_pipeline import embed_and_store
    # embed_and_store(all_docs)
   

Processing 3GPP documents:   8%|▊         | 1/12 [00:20<03:46, 20.62s/it]


[INFO] 37340-h30.docx — raw length: 365,911 chars
       After cleaning: 355,038 chars
  → 435 chunks after filtering (min 50 words)
       Created 435 chunks


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1031 > 512). Running this sequence through the model will result in indexing errors



[INFO] 38211-h30.docx — raw length: 1,309,130 chars
       After cleaning: 1,291,116 chars


Processing 3GPP documents:  17%|█▋        | 2/12 [02:24<13:33, 81.31s/it]

...


[INFO] 38322-h30.docx — raw length: 72,320 chars
       After cleaning: 65,350 chars
  → 89 chunks after filtering (min 50 words)
       Created 89 chunks


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (3318 > 512). Running this sequence through the model will result in indexing errors
Processing 3GPP documents:  75%|███████▌  | 9/12 [18:25<03:53, 77.90s/it] 


[INFO] 38323-h30.docx — raw length: 103,245 chars
       After cleaning: 102,854 chars
  → 138 chunks after filtering (min 50 words)
       Created 138 chunks


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (635 > 512). Running this sequence through the model will result in indexing errors
Processing 3GPP documents:  75%|███████▌  | 9/12 [41:11<13:43, 274.59s/it]


KeyboardInterrupt: 

'6.1 MAC Sublayer\nIn MR-DC, the UE is configured with two MAC entities: one MAC entity for the MCG and one MAC entity for the SCG. The serving cells other than the PCell can be activated/deactivated by RRC or MAC Control Element. For activation/deactivation by MAC Control Element, the serving cells of the MCG other than the PCell can only be activated/deactivated by the MAC Control Element received on MCG, and the serving cells of the SCG other than PSCell can only be activated/ deactivated by the MAC Control Element received on SCG. The MAC entity applies the bitmap for the associated cells of either MCG or SCG. When the SCG is not deactivated, the PSCell is always activated like the PCell (i.e. deactivation timer is not applied to PSCell). With the exception of PUCCH SCell, one deactivation timer is configured per SCell by RRC.\nIn MR-DC, semi-persistent scheduling (SPS) resources and configured grant (CG) resources can be configured on serving cells in both MCG and SCG.'

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  EMBEDDING & VECTOR STORE PIPELINE
# ─────────────────────────────────────────────────────────────────────────────
from pathlib import Path
from typing import Optional, List
import logging
import time
import psutil
from tqdm import tqdm
import numpy as np
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings

# ─────────────────────────────────────────────────────────────────────────────
#  CONSTANTS
# ─────────────────────────────────────────────────────────────────────────────
PROJECT_ROOT      = Path.cwd().parent
VECTOR_STORE_DIR = PROJECT_ROOT / "data" / "vectorstore"
FAISS_INDEX_PATH  = VECTOR_STORE_DIR / "faiss_index"

# Model config
MODEL_NAME          = "BAAI/bge-small-en"  # 384-dim. Lightweight & effective.
MODEL_DEVICE        = "cpu"                # Use "cuda" if GPU available.
NORMALIZE_EMBEDDINGS= True                 # Recommended for cosine similarity.
BATCH_SIZE          = 128                  # Optimal for memory/throughput tradeoff.

# Thresholds
MIN_EMBEDDING_DIM   = 100                  # Sanity check for model output.
MAX_MEMORY_USAGE    = 0.85                 # Halt if RAM exceeds 85%.
EMBEDDING_TIMEOUT   = 300                  # 5 min timeout per batch.

# Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s — %(levelname)s — %(message)s",
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger(__name__)

# ─────────────────────────────────────────────────────────────────────────────
#  UTILITIES
# ─────────────────────────────────────────────────────────────────────────────
def check_memory_usage() -> bool:
    """Return True if memory usage is below MAX_MEMORY_USAGE."""
    return psutil.virtual_memory().percent < MAX_MEMORY_USAGE * 100

def validate_embeddings(embeddings: List[List[float]]) -> bool:
    """Sanity-check embedding batch."""
    if not embeddings:
        return False
    dim = len(embeddings[0])
    if dim < MIN_EMBEDDING_DIM:
        logger.error(f"Embedding dimension {dim} < {MIN_EMBEDDING_DIM} — invalid model output.")
        return False
    for emb in embeddings:
        if len(emb) != dim:
            logger.error("Inconsistent embedding dimensions in batch.")
            return False
        if np.all(emb == 0):  # Zero vector
            logger.warning("Zero-vector embedding detected — may indicate model failure.")
    return True

def get_embedding_model() -> Embeddings:
    """Initialize and validate the embedding model."""
    start = time.time()
    logger.info(f"Loading embedding model: {MODEL_NAME} (device={MODEL_DEVICE})...")

    embeddings = HuggingFaceEmbeddings(
        model_name=MODEL_NAME,
        model_kwargs={"device": MODEL_DEVICE},
        encode_kwargs={"normalize_embeddings": NORMALIZE_EMBEDDINGS},
    )

    # Test with a dummy text
    test_emb = embeddings.embed_query("test")
    if not validate_embeddings([test_emb]):
        raise RuntimeError("Embedding model validation failed.")

    logger.info(f"Model loaded in {time.time() - start:.2f}s.")
    return embeddings

# ─────────────────────────────────────────────────────────────────────────────
#  CORE EMBEDDING & STORAGE
# ─────────────────────────────────────────────────────────────────────────────
def embed_documents(
    docs: List[Document],
    embeddings: Embeddings,
    batch_size: int = BATCH_SIZE,
) -> List[List[float]]:
    """
    Embed a list of documents in batches with memory checks.
    Returns list of embedding vectors (same order as input docs).
    """
    all_embeddings = []
    for i in tqdm(
        range(0, len(docs), batch_size),
        desc="Embedding batches",
        unit="batch",
    ):
        batch = docs[i : i + batch_size]
        if not check_memory_usage():
            logger.warning("Memory usage high — pausing for GC...")
            import gc
            gc.collect()
            time.sleep(5)

        try:
            batch_embeddings = embeddings.embed_documents(
                [d.page_content for d in batch]
            )
            if not validate_embeddings(batch_embeddings):
                raise ValueError("Invalid embeddings in batch.")
            all_embeddings.extend(batch_embeddings)
        except Exception as e:
            logger.error(f"Failed to embed batch {i//batch_size}: {e}")
            raise
    return all_embeddings

def build_vector_store(
    docs: List[Document],
    embeddings: Embeddings,
    save_path: Path,
    merge_existing: bool = True,
) -> FAISS:
    """
    Build and save a FAISS index from documents.
    If `merge_existing=True`, loads and merges with any existing index at `save_path`.
    """
    save_path.parent.mkdir(parents=True, exist_ok=True)

    if merge_existing and save_path.exists():
        logger.info(f"Loading existing index from {save_path} for incremental update...")
        existing_store = FAISS.load_local(
            str(save_path),
            embeddings=embeddings,
            allow_dangerous_deserialization=True,
        )
        logger.info(f"Existing index has {len(existing_store.docstore._dict)} documents.")
    else:
        existing_store = None

    # Embed all new documents
    logger.info(f"Embedding {len(docs)} new documents...")
    new_embeddings = embed_documents(docs, embeddings)

    # Create new FAISS index for the batch
    new_index = FAISS.from_documents(
        documents=docs,
        embedding=embeddings,
    )
    new_index.index = new_index.index  # Ensure index is built

    if existing_store:
        # Merge indices (FAISS does not natively support incremental adds with metadata)
        # Workaround: Rebuild combined index
        combined_docs = existing_store.docstore._dict.values()
        combined_docs = list(combined_docs) + docs
        combined_embeddings = [
            existing_store.embedding_function.embed_query(doc.page_content)
            for doc in combined_docs
        ]
        vector_store = FAISS.from_embeddings(
            embeddings=combined_embeddings,
            documents=combined_docs,
            embedding=embeddings,
        )
    else:
        vector_store = new_index

    # Save
    vector_store.save_local(str(save_path))
    logger.info(f"Vector store saved to {save_path} (total docs: {len(vector_store.docstore._dict)}).")
    return vector_store

# ─────────────────────────────────────────────────────────────────────────────
#  VALIDATION & DIAGNOSTICS
# ─────────────────────────────────────────────────────────────────────────────
def validate_vector_store(store: FAISS, sample_size: int = 5) -> None:
    """Spot-check the vector store for integrity."""
    if len(store.docstore._dict) == 0:
        logger.error("Vector store is empty!")
        return

    # Check embedding dimensions
    sample_docs = list(store.docstore._dict.values())[:sample_size]
    for doc in sample_docs:
        emb = store.embedding_function.embed_query(doc.page_content)
        if len(emb) < MIN_EMBEDDING_DIM:
            logger.error(f"Invalid embedding dimension for doc: {doc.metadata.get('section')}")

    # Check metadata
    missing_meta = [d for d in sample_docs if "section" not in d.metadata]
    if missing_meta:
        logger.warning(f"{len(missing_meta)} docs missing 'section' metadata in sample.")

    logger.info(f"Vector store validation passed (sampled {sample_size} docs).")

# ─────────────────────────────────────────────────────────────────────────────
#  MAIN PIPELINE
# ─────────────────────────────────────────────────────────────────────────────
def embed_and_store(
    all_docs: List[Document],
    vector_store_path: Path = FAISS_INDEX_PATH,
    merge_existing: bool = True,
) -> FAISS:
    """
    End-to-end pipeline:
    1. Initialize embedding model.
    2. Embed documents in batches.
    3. Build/merge FAISS index.
    4. Save to disk.
    5. Validate.
    """
    if not all_docs:
        logger.warning("No documents to embed — skipping.")
        return None

    embeddings = get_embedding_model()
    vector_store = build_vector_store(
        all_docs,
        embeddings,
        vector_store_path,
        merge_existing=merge_existing,
    )
    validate_vector_store(vector_store)
    return vector_store

# ─────────────────────────────────────────────────────────────────────────────
#  ENTRY POINT (INTEGRATION WITH YOUR EXISTING CODE)
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    # ── Load and chunk documents (your existing code) ──
    from your_chunking_module import load_and_chunk_3gpp_docs, validate_chunks

    three_gpp_dir = PROJECT_ROOT / "data" / "raw" / "3gpp_docs"
    all_docs = load_and_chunk_3gpp_docs(three_gpp_dir, glob_pattern="*.docx")
    validate_chunks(all_docs)

    # ── Embed and store ──
    vector_store = embed_and_store(
        all_docs,
        vector_store_path=FAISS_INDEX_PATH,
        merge_existing=True,  # Set False to overwrite
    )